# Research QuantBook: MomentumStrategy (Sector ETF Rotation)

## Objectif
Reproduire l'analyse exploratoire de `research.ipynb` avec les donnees natives QuantConnect.

## Performance actuelle
- **Sharpe**: 0.472, **CAGR**: 11.8%, **MaxDD**: 25.8%
- **Signal**: Vol-adjusted momentum (12m-1m return / 63d vol)
- **Univers**: 11 ETFs sectoriels GICS (XLK, XLF, XLE, XLV, XLI, XLY, XLP, XLU, XLB, XLRE, XLC)
- **Regime filter**: SPY < SMA200 AND SMA20 -> rotation defensive XLP+XLU

## Hypotheses a tester
1. Top-N sensitivity (2, 3, 4, 5, 6 positions)
2. Lookback period (6m, 9m, 12m, 18m)
3. Vol window (20d, 40d, 63d, 90d)
4. Stop-loss threshold (-8%, -10%, -12%, -15%)
5. Regime filter variants (SMA200 only, SMA200+SMA20, aucun)

## Prerequis
- Environnement Lean Research
- Duree estimee: ~5 minutes

In [ ]:
# Setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

qb = QuantBook()
print("QuantBook initialise.")

## 1. Chargement des donnees

11 ETFs sectoriels GICS + SPY (benchmark et regime filter).

In [ ]:
sector_etfs = ['XLK', 'XLF', 'XLE', 'XLV', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLRE', 'XLC']
all_tickers = sector_etfs + ['SPY']

symbols = {}
for ticker in all_tickers:
    symbols[ticker] = qb.add_equity(ticker, Resolution.DAILY).symbol

start = datetime(2015, 1, 1)
end = datetime(2026, 1, 1)

history = qb.history(list(symbols.values()), start, end, Resolution.DAILY)
closes = history['close'].unstack(level=0)

symbol_to_ticker = {str(v): k for k, v in symbols.items()}
closes.columns = [symbol_to_ticker.get(str(c), str(c)) for c in closes.columns]
closes = closes.dropna()

print(f"Periode: {closes.index[0].date()} a {closes.index[-1].date()}")
print(f"Donnees: {len(closes)} jours de trading")
print(f"Secteurs: {list(closes.columns)}")

returns_df = closes.pct_change()

## 2. Fonctions de backtest

In [ ]:
def momentum_score(prices, lookback=252, skip=21, vol_window=63):
    """Score momentum ajuste par la volatilite (Jegadeesh skip-month)."""
    raw_mom = prices.shift(skip) / prices.shift(lookback) - 1
    vol = prices.pct_change().rolling(vol_window).std() * np.sqrt(252)
    return raw_mom / vol

def backtest_sector_momentum(closes, sector_etfs, top_n=4, lookback=252, skip=21,
                              vol_window=63, stop_loss=-0.10,
                              regime_filter='both', sma_period=200):
    """Backtest rotation sectorielle momentum."""
    returns_df = closes.pct_change()
    spy = closes['SPY']
    sma200 = spy.rolling(sma_period).mean()
    sma20 = spy.rolling(20).mean()
    
    # Calculer les scores momentum pour chaque secteur
    scores = pd.DataFrame(index=closes.index)
    for etf in sector_etfs:
        if etf in closes.columns:
            scores[etf] = momentum_score(closes[etf], lookback, skip, vol_window)
    
    # Rebalancement mensuel
    monthly = closes.resample('MS').first().index
    positions = pd.DataFrame(0.0, index=closes.index, columns=sector_etfs)
    entry_prices = {}
    
    for i, date in enumerate(monthly):
        if date not in closes.index:
            continue
        if scores.loc[date].isna().all():
            continue
        
        # Regime filter
        if regime_filter == 'both':
            bear = spy[date] < sma200[date] and spy[date] < sma20[date]
        elif regime_filter == 'sma200':
            bear = spy[date] < sma200[date]
        else:
            bear = False
        
        if bear:
            # Defensive: XLP + XLU equal weight
            w = 1.0 / 2
            current = pd.Series(0.0, index=sector_etfs)
            if 'XLP' in sector_etfs: current['XLP'] = w
            if 'XLU' in sector_etfs: current['XLU'] = w
        else:
            # Top-N momentum
            s = scores.loc[date].dropna()
            top = s.nlargest(top_n).index.tolist()
            current = pd.Series(0.0, index=sector_etfs)
            w = 1.0 / top_n
            for etf in top:
                current[etf] = w
        
        next_date = monthly[i + 1] if i + 1 < len(monthly) else closes.index[-1]
        positions.loc[date:next_date] = current.values
        for etf in sector_etfs:
            if current[etf] > 0:
                entry_prices[etf] = closes[etf][date]
    
    # Calculer les rendements du portefeuille
    port_ret = (positions.shift(1) * returns_df[sector_etfs]).sum(axis=1)
    port_ret = port_ret[port_ret.index >= monthly[0]]
    
    return port_ret

def calc_stats(returns, name=''):
    total = (1 + returns).cumprod().iloc[-1] - 1
    years = len(returns) / 252
    cagr = (1 + total) ** (1 / years) - 1
    vol = returns.std() * np.sqrt(252)
    sharpe = (cagr - 0.03) / vol if vol > 0 else 0
    cum = (1 + returns).cumprod()
    dd = (cum / cum.cummax() - 1).min()
    return {'name': name, 'CAGR': f'{cagr:.1%}', 'Vol': f'{vol:.1%}',
            'Sharpe': f'{sharpe:.3f}', 'MaxDD': f'{dd:.1%}'}

print("Fonctions definies.")

## 3. Hypothese 1: Top-N positions

Tester la concentration du portefeuille (2 a 6 positions).

In [ ]:
results_topn = []
for n in [2, 3, 4, 5, 6]:
    ret = backtest_sector_momentum(closes, sector_etfs, top_n=n)
    results_topn.append(calc_stats(ret, f'Top-{n}'))

spy_ret = closes['SPY'].pct_change().dropna()
results_topn.append(calc_stats(spy_ret, 'SPY B&H'))

print("=== H1: Top-N positions ===")
print(pd.DataFrame(results_topn).set_index('name').to_string())

### Verdict H1

Top-4 est le parametre actuel. Verifier si Top-3 (plus concentre) ou Top-5 (plus diversifie)
offre un meilleur compromis Sharpe/MaxDD.

## 4. Hypothese 2: Lookback period

Tester differentes fenetres de lookback pour le calcul du momentum.

In [ ]:
results_lb = []
for lb_months, lb_days in [(6, 126), (9, 189), (12, 252), (18, 378)]:
    ret = backtest_sector_momentum(closes, sector_etfs, lookback=lb_days)
    results_lb.append(calc_stats(ret, f'{lb_months}m ({lb_days}d)'))

print("=== H2: Lookback period ===")
print(pd.DataFrame(results_lb).set_index('name').to_string())

### Verdict H2

12m est le lookback canonique (Jegadeesh & Titman). Verifier la robustesse
sur donnees QC vs yfinance.

## 5. Hypothese 3: Vol window

La fenetre de volatilite pour l'ajustement du score momentum.

In [ ]:
results_vol = []
for vw in [20, 40, 63, 90]:
    ret = backtest_sector_momentum(closes, sector_etfs, vol_window=vw)
    results_vol.append(calc_stats(ret, f'Vol {vw}d'))

print("=== H3: Vol window ===")
print(pd.DataFrame(results_vol).set_index('name').to_string())

### Verdict H3

Regle #9 du backlog: Vol window 60d > 20d. Verifier si 63d reste optimal.

## 6. Hypothese 4: Regime filter

Comparer: aucun filtre, SMA200 seul, SMA200+SMA20 (actuel).

In [ ]:
results_regime = []
for name, rf in [('Aucun', 'none'), ('SMA200 only', 'sma200'), ('SMA200+SMA20', 'both')]:
    ret = backtest_sector_momentum(closes, sector_etfs, regime_filter=rf)
    results_regime.append(calc_stats(ret, name))

print("=== H4: Regime filter ===")
print(pd.DataFrame(results_regime).set_index('name').to_string())

### Verdict H4

Le double filtre SMA200+SMA20 evite les faux signaux bear. Verifier si
le gain en MaxDD compense la perte de rendement.

## 7. Visualisation

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# H1: Top-N
ax = axes[0, 0]
for n in [2, 3, 4, 5, 6]:
    ret = backtest_sector_momentum(closes, sector_etfs, top_n=n)
    cum = (1 + ret).cumprod()
    ax.plot(cum.index, cum, label=f'Top-{n}', linewidth=1.5)
ax.set_title('H1: Top-N positions', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# H2: Lookback
ax = axes[0, 1]
for lb_m, lb_d in [(6, 126), (9, 189), (12, 252), (18, 378)]:
    ret = backtest_sector_momentum(closes, sector_etfs, lookback=lb_d)
    cum = (1 + ret).cumprod()
    ax.plot(cum.index, cum, label=f'{lb_m}m', linewidth=1.5)
ax.set_title('H2: Lookback period', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# H3: Vol window
ax = axes[1, 0]
for vw in [20, 40, 63, 90]:
    ret = backtest_sector_momentum(closes, sector_etfs, vol_window=vw)
    cum = (1 + ret).cumprod()
    ax.plot(cum.index, cum, label=f'Vol {vw}d', linewidth=1.5)
ax.set_title('H3: Vol window', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# H4: Regime filter
ax = axes[1, 1]
for name, rf in [('Aucun', 'none'), ('SMA200', 'sma200'), ('SMA200+SMA20', 'both')]:
    ret = backtest_sector_momentum(closes, sector_etfs, regime_filter=rf)
    cum = (1 + ret).cumprod()
    ax.plot(cum.index, cum, label=name, linewidth=1.5)
spy_cum = (1 + spy_ret).cumprod()
ax.plot(spy_cum.index, spy_cum, label='SPY B&H', linestyle='--', alpha=0.5)
ax.set_title('H4: Regime filter', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('momentumstrategy_quantbook_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Conclusions

### Tableau recapitulatif

| Hypothese | Resultat QuantBook | Coherent avec yfinance? |
|-----------|-------------------|-------------------------|
| H1 Top-N | (a remplir) | (a verifier) |
| H2 Lookback | (a remplir) | (a verifier) |
| H3 Vol window | (a remplir) | (a verifier) |
| H4 Regime filter | (a remplir) | (a verifier) |

### Regles du backlog appliquees

- Regle #1: Risk-adjusted momentum confirme
- Regle #2: Skip-month (Jegadeesh 1990)
- Regle #4: Stop-loss -10% pour equities
- Regle #9: Vol window 60d > 20d
- Regle #17: Divergence yfinance documentee